# 6e Visualization feature dashboards

## 1 Setup

### Load libraries

In [44]:
library(readxl)
library(tidyverse)
library(RColorBrewer)
library(ggbeeswarm)
library(scales)
library(patchwork)
library(ggtext)
options(warn = -1)


### Setup workspace

In [ ]:
# Set working directory to project folder
wd <- "/path/to/02_metabolomics"
setwd(wd)

### Define functions  


In [46]:
# Generate breaks focused around upper magnitudes of a numeric range
calculate_focused_breaks <- function(range) {
  log_max <- ceiling(log10(max(range, na.rm = TRUE)))
  highest_power <- 10^log_max
  second_highest_power <- 10^(log_max - 1)

  highest_breaks <- seq(from = highest_power, by = -highest_power / 10, length.out = 10)[10:1]
  highest_breaks <- highest_breaks[highest_breaks <= max(range) & highest_breaks >= second_highest_power]

  second_highest_breaks <- seq(from = second_highest_power, by = -second_highest_power / 2, length.out = 2)[10:1]
  second_highest_breaks <- second_highest_breaks[second_highest_breaks < highest_power]

  third_power <- 10^(log_max - 2)

  combined_breaks <- unique(c(second_highest_breaks, highest_breaks, third_power))
  sort(combined_breaks[combined_breaks >= min(range) & combined_breaks <= max(range)])
}

# Format scientific y-axis labels in ggplot2 using TeX-style notation
scientific_10 <- function(x) {
  labels <- scales::scientific_format()(x)
  labels <- gsub("e\\+0*$", "", labels)
  labels <- gsub("e\\+", "e", labels)
  labels <- gsub("e", " %*% 10^", labels)

  labels <- ifelse(grepl("%*%", labels), labels, paste0(labels, ""))

  sapply(labels, function(l) {
    if (grepl("%*%", l)) parse(text = l) else as.expression(l)
  })
}

# Label mapping for ggplot2 facets
labeller <- function(variable, value) {
  return(rename_dict[value])
}

labeller_pks <- function(variable, value) {
  return(rename_dict_pks[value])
}

# Add leading zeros to numeric strings
pad_number_float_string <- function(number_string, total_digits) {
  number_as_integer <- as.integer(as.numeric(number_string))
  sprintf(paste0("%0", total_digits, "d"), number_as_integer)
}

# Convert decimal minutes to a "mm:ss" formatted string
convert_to_min_sec <- function(minutes) {
  whole_minutes <- floor(minutes)
  seconds <- round((minutes - whole_minutes) * 60)
  sprintf("%02dm%02ds", whole_minutes, seconds)
}


### Define colors and other parameters

In [197]:
# Color palettes for different groups
wt_colors      <- brewer.pal(n = 9, name = "Greens")
disfp_colors   <- brewer.pal(n = 9, name = "Purples")
rescue_colors  <- brewer.pal(n = 9, name = "Blues")
pks_colors     <- brewer.pal(n = 9, name = "OrRd")

color_nr <- 6  # Index for consistent shade selection

# Main color mapping for sample groups
group_colors <- c(
  "wildtype_veg" = wt_colors[color_nr],
  "wildtype_stv" = wt_colors[color_nr],
  "wildtype_fbs" = wt_colors[color_nr],
  "disfp_veg" = disfp_colors[color_nr],
  "disfp_stv" = disfp_colors[color_nr],
  "disfp_fbs" = disfp_colors[color_nr],
  "wildtype2_axenic_fbs" = wt_colors[color_nr],
  "pks5_fbs" = pks_colors[color_nr],
  "pks7_fbs" = pks_colors[color_nr],
  "pks24_fbs" = pks_colors[color_nr],
  "pks25_fbs" = pks_colors[color_nr],
  "pks24pks25_fbs" = pks_colors[color_nr],
  "blank" = "azure3",
  "pblank" = "azure2",
  "KA" = "darkgoldenrod",
  "HL5" = "darkorange4",
  "SM5" = "darkorange4"
)

# Lighter color variant for some plots
group_colors_light <- c(
  "wildtype_veg" = wt_colors[color_nr],
  "wildtype_stv" = wt_colors[color_nr],
  "wildtype_fbs" = wt_colors[color_nr],
  "disfp_veg" = disfp_colors[color_nr],
  "disfp_stv" = disfp_colors[color_nr],
  "disfp_fbs" = disfp_colors[color_nr],
  "wildtype2_axenic_fbs" = wt_colors[color_nr],
  "pks5_fbs" = pks_colors[color_nr],
  "pks7_fbs" = pks_colors[color_nr],
  "pks24_fbs" = pks_colors[color_nr],
  "pks25_fbs" = pks_colors[color_nr],
  "pks24pks25_fbs" = pks_colors[color_nr],
  "blank" = "azure1",
  "pblank" = "azure1",
  "KA" = "darkgoldenrod1",
  "HL5" = "darkorange1",
  "SM5" = "darkorange1"
)

# Dictionaries for renaming variables in plots
rename_mode_dict <- list(
  "pos" = "positive",
  "neg" = "negative"
)

rename_change_dict <- list(
  "up" = "Increased level",
  "down" = "Decreased level",
  "abs" = "Absent"
)

rename_stage_dict <- list(
  "veg" = "vegetative cells",
  "stv" = "starved cells",
  "fbs" = "fruiting bodies"
)

rename_dict <- list(
  "wildtype_veg" = expression("wild-type"),
  "disfp_veg" = expression(italic(disfp)^"–"),
  "wildtype_stv" = expression("wild-type"),
  "disfp_stv" = expression(italic(disfp)^"–"),
  "wildtype_fbs" = expression("wild-type"),
  "disfp_fbs" = expression(italic(disfp)^"–"),
  "pblank" = expression(atop("process blank", "control")),
  "blank" = expression(atop("blank", "control")),
  "KA" = expression(atop(italic(Klebsiella ~ aerogenes), "control")),
  "SM5" = expression(atop(SM5 ~ "medium", "control"))
)

rename_dict_pks <- list(
  "wildtype2_axenic_fbs" = expression("wild-type"),
  "pks5_fbs" = expression(italic(pks5)^"–"),
  "pks7_fbs" = expression(italic(pks7)^"–"),
  "pks24_fbs" = expression(italic(pks24)^"–"),
  "pks25_fbs" = expression(italic(pks25)^"–"),
  "pks24pks25_fbs" = expression(italic(pks24)^"–" * "/" * italic(pks25)^"–")
)


rename_dict_pks_intersection <- list(
  "pks-related" = " · **Intersection:** <i>pks</i>-related",
  "disfp" = " · **Intersection:** <i>disfp</i><sup>–</sup>-specific",
  "pks5" = " · **Intersection:** <i>pks5</i><sup>–</sup>",
  "pks7" = " · **Intersection:** <i>pks7</i><sup>–</sup>",
  "pks24" = " · **Intersection:**  <i>pks24</i><sup>–</sup>",
  "pks25" = " · **Intersection:**  <i>pks25</i><sup>–</sup>",
  "pks24_pks25" = " · **Intersection:**  <i>pks24</i><sup>–</sup> and <i>pks25</i><sup>–</sup>",
  "pks2425" = " · **Intersection:**  <i>pks24</i><sup>–</sup>/<i>pks25</i><sup>–</sup>",
  "pks24_pks25_pks2425" = " · **Intersection:**  <i>pks24</i><sup>–</sup>, <i>pks25</i><sup>–</sup>, <i>pks24</i><sup>–</sup>/<i>pks25</i><sup>–</sup>"
)


## 2 Load data

In [48]:
# Load sample metadata
md_path <- "02_data/tables/samplelist.xlsx"
md.load <- read_excel(md_path)
md <- md.load %>% filter(type %in% c("sample", "control"))

# Load group definitions
mdgroups_path <- "02_data/tables/grouplist.xlsx"
mdgroups <- read_excel(mdgroups_path)

### Load feature table

In [49]:
ft_filePath <- file.path(wd, "03_analysis", "tables", "featureTable.csv")
ft <- read.csv(ft_filePath)

### Preparing stage-specific feature tables  
Creates a function to prepare a feature table for each developmental stage (vegetative, starved, fruiting body),  
selecting relevant columns, renaming them for consistency, and combining all stages into a single table.


In [50]:
# Function to extract and clean feature data for a specific stage
prepare_stage_table <- function(stage_name) {
  change_col   <- paste0("change_", stage_name)
  wt_mean_col  <- paste0(stage_name, "_wt_mean")
  pvalue_col   <- paste0(stage_name, "_pvalue")
  log2fc_col   <- paste0(stage_name, "_log2FC")
  
  ft %>%
    filter(.data[[change_col]] != "na") %>%
    select(
      feature, mode, mz, rt, rt_range.min, rt_range.max,
      sirius_smiles, sirius_molecularFormula,
      sirius_NPC.pathway, sirius_NPC.superclass, sirius_NPC.class,
      sirius_ClassyFire.superclass, sirius_ClassyFire.class, sirius_ClassyFire.subclass,
      metaChange_pks, msms,
      !!sym(change_col), !!sym(wt_mean_col), !!sym(pvalue_col), !!sym(log2fc_col)
    ) %>%
    mutate(stage = stage_name) %>%
    rename(
      change  = !!sym(change_col),
      wt_mean = !!sym(wt_mean_col),
      pvalue  = !!sym(pvalue_col),
      log2FC  = !!sym(log2fc_col)
    )
}

# Prepare individual stage tables
ft_veg <- prepare_stage_table("veg")
ft_stv <- prepare_stage_table("stv")
ft_fb  <- prepare_stage_table("fbs")

# Combine all into a single table
ft_changes <- bind_rows(ft_veg, ft_stv, ft_fb)

# Add ranking per stage and change category
ft_changes <- ft_changes %>%
  group_by(stage, change) %>%
  mutate(
    rank_wt_mean = rank(-wt_mean, ties.method = "first"),
    rank_pvalue  = rank(pvalue, ties.method = "first"),
    rank = if_else(
      change == "abs",
      rank_wt_mean,  # Use intensity rank for "abs"
      rank_pvalue    # Use p-value rank otherwise
    )
  ) %>%
  ungroup()

# Select only up/down/abs features
ft_selected <- ft_changes %>%
  filter(change %in% c("up", "down", "abs")) #%>% slice_head(n = 1)




## Draw feature dashboards

Generates a visual dashboard for each deregulated feature, summarizing its key characteristics and behavior across conditions.  
Each dashboard includes:

- **EIC plots** for both xenic and axenic samples (if matched to pks mutant),  
- a **boxplot** of peak areas across relevant strain groups,  
- **feature metadata** such as m/z, retention time, scan mode, and statistical values,  
- **annotations** from SIRIUS inclidung NPClassifier, and ClassyFire, and  
- the **predicted chemical structure** and a **name** (if available).

The plots are automatically saved to the results directory and organized by stage, change type, and compound class.


### Define functions for the different plots

In [51]:
get_dashboard_metadata <- function(feature_selected, stage_selected, ft, ft_changes) {
  # Pull values from ft and ft_changes
  mode_selected       <- ft %>% filter(feature == feature_selected) %>% pull(mode)
  msms_selected       <- ft %>% filter(feature == feature_selected) %>% pull(msms)
  rank_selected       <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(rank)
  change_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(change)
  pks_intersection    <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(metaChange_pks)
  log2FC_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(log2FC)
  pvalue_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(pvalue)

  rt_selected         <- ft %>% filter(feature == feature_selected) %>% pull(rt)
  mz_selected         <- ft %>% filter(feature == feature_selected) %>% pull(mz)
  rtMin               <- ft %>% filter(feature == feature_selected) %>% pull(rt_range.min)
  rtMin_plot          <- max(rtMin - 5, 0)
  rtMin_plot          <- rtMin
  rtMax               <- ft %>% filter(feature == feature_selected) %>% pull(rt_range.max)
  rtMax_plot          <- min(rtMax + 5, 12)
  rtMax_plot          <- rtMax

  CF_Superclass       <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.superclass")
  CF_Class            <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.class")
  CF_Subclass         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.subclass")
  CF_Level5           <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.level.5")
  CF_MostSpecific     <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.most.specific.class")

  NPC_Pathway         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.pathway")
  NPC_Superclass      <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.superclass")
  NPC_Class           <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.class")

  Sirius_Name         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_name")
  Sirius_Formula      <- ft %>% filter(feature == feature_selected) %>% pull("sirius_molecularFormula")
  Sirius_Smiles       <- ft %>% filter(feature == feature_selected) %>% pull("sirius_smiles")

  # Return as a named list
  return(list(
    feature = feature_selected,
    stage = stage_selected,
    mode = mode_selected,
    msms = msms_selected,
    rank = rank_selected,
    change = change_selected,
    metaChange_pks = pks_intersection,
    log2FC = log2FC_selected,
    pvalue = pvalue_selected,
    rt = rt_selected,
    mz = mz_selected,
    rtMin = rtMin,
    rtMin_plot = rtMin_plot,
    rtMax = rtMax,
    rtMax_plot = rtMax_plot,
    CF_Superclass = CF_Superclass,
    CF_Class = CF_Class,
    CF_Subclass = CF_Subclass,
    CF_Level5 = CF_Level5,
    CF_MostSpecific = CF_MostSpecific,
    NPC_Pathway = NPC_Pathway,
    NPC_Superclass = NPC_Superclass,
    NPC_Class = NPC_Class,
    Sirius_Name = Sirius_Name,
    Sirius_Formula = Sirius_Formula,
    smiles = Sirius_Smiles
  ))
}


In [52]:
make_eic_plot_xenic <- function(feature_selected, meta, md, wd) {
  eic_path <- file.path(wd, "03_analysis", "eic", paste0(feature_selected, ".csv"))
  eic_data <- read.csv(eic_path) %>%
    rename(filename = File, rtime = RT, intensity = Intensity)

  eic_data$filename <- str_replace_all(eic_data$filename, ".mzML", "")
  eic_data <- merge(eic_data, md, by.x = "filename", by.y = "name", all.x = TRUE)

  eic_filtered <- eic_data %>%
    filter(between(rtime / 60, meta$rtMin_plot, meta$rtMax_plot)) %>%
    filter(type == "sample", cultivation == "xenic") %>%
    filter(replicate %in% c(1, 2, 3, 7, 8, 9, 13, 14, 15)) %>%
    filter(stage == meta$stage)

  eic_filtered$strain_group <- factor(
    eic_filtered$strain_group,
    levels = c("wildtype_veg", "disfp_veg", "wildtype_stv", "disfp_stv", "wildtype_fbs", "disfp_fbs", "pblank", "blank", "KA", "SM5", "HL5")
  )

  dynamic_breaks <- calculate_focused_breaks(range(eic_filtered$intensity))
  max_int <- eic_filtered %>%
    filter(between(rtime / 60, meta$rtMin, meta$rtMax)) %>%
    pull(intensity) %>% max()

  plot <- eic_filtered %>% arrange(strain_group) %>%
    ggplot(aes(x = rtime / 60, y = intensity, group = filename, fill = strain_group, color = strain_group)) +
    geom_segment(aes(x = meta$rt, xend = meta$rt, y = 0, yend = Inf), color = "darkgrey", linetype = "solid") +
    geom_segment(aes(x = meta$rtMin, xend = meta$rtMin, y = 0, yend = Inf), color = "grey80", linetype = "dashed") +
    geom_segment(aes(x = meta$rtMax, xend = meta$rtMax, y = 0, yend = Inf), color = "grey80", linetype = "dashed") +
    geom_ribbon(aes(ymin = 0, ymax = intensity), alpha = 0.2) +
    geom_line() +
    labs(x = "retention time in min", y = "intensity", color = "genotype") +
    scale_fill_manual(values = group_colors_light) +
    scale_color_manual(values = group_colors) +
    guides(color = FALSE, fill = FALSE) +
    scale_y_continuous(label = scientific_10, breaks = dynamic_breaks) +
    coord_cartesian(xlim = c(meta$rtMin_plot, meta$rtMax_plot), ylim = c(NA, max_int)) +
    theme_minimal(base_size = 12) +
    facet_wrap(~strain_group, nrow = 1, labeller = labeller) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = .99, vjust = -20, face = "bold", color = "grey40")
    )

  return(plot)
}


In [53]:
make_eic_plot_axenic <- function(feature_selected, meta, md, wd) {
  eic_path <- file.path(wd, "03_analysis", "eic", paste0(feature_selected, ".csv"))
  eic_data <- read.csv(eic_path) %>%
    rename(filename = File, rtime = RT, intensity = Intensity)

  eic_data$filename <- str_replace_all(eic_data$filename, ".mzML", "")
  eic_data <- merge(eic_data, md, by.x = "filename", by.y = "name", all.x = TRUE)

  eic_filtered <- eic_data %>%
    filter(between(rtime / 60, meta$rtMin_plot, meta$rtMax_plot)) %>%
    filter(type == "sample", cultivation == "axenic", strain_group != "wildtype1_axenic_fbs") %>%
    filter(replicate %in% c(1, 2, 3)) %>%
    filter(stage == "fbs")

  eic_filtered$strain_group <- factor(
    eic_filtered$strain_group,
    levels = c("wildtype2_axenic_fbs", "pks5_fbs", "pks7_fbs", "pks24_fbs", "pks25_fbs", "pks24pks25_fbs")
  )

  dynamic_breaks <- calculate_focused_breaks(range(eic_filtered$intensity))
  max_int <- eic_filtered %>%
    filter(between(rtime / 60, meta$rtMin, meta$rtMax)) %>%
    pull(intensity) %>% max()

  plot <- eic_filtered %>% arrange(strain_group) %>%
    ggplot(aes(x = rtime / 60, y = intensity, group = filename, fill = strain_group, color = strain_group)) +
    geom_vline(xintercept = meta$rt, linetype = "solid", color = "darkgrey") +
    geom_ribbon(aes(ymin = 0, ymax = intensity), alpha = 0.2) +
    geom_line() +
    labs(x = "retention time in min", y = "intensity", color = "genotype") +
    scale_fill_manual(values = group_colors_light) +
    scale_color_manual(values = group_colors) +
    guides(color = FALSE, fill = FALSE) +
    scale_y_continuous(label = scientific_10, breaks = dynamic_breaks) +
    coord_cartesian(xlim = c(meta$rtMin_plot, meta$rtMax_plot), ylim = c(NA, max_int)) +
    theme_minimal(base_size = 12) +
    facet_wrap(~strain_group, nrow = 1, labeller = labeller_pks) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = .99, vjust = -20, face = "bold", color = "grey40")
    )

  return(plot)
}


In [54]:
make_boxplot <- function(feature_selected, stage_selected, ft, md) {
  md.select <- md %>%
    filter(type == "sample", cultivation == "xenic", stage == stage_selected)
  
  name <- md.select %>% pull(name)

  quant_target <- ft %>%
    filter(feature == feature_selected) %>%
    .[, name] %>%
    pivot_longer(cols = name, names_to = "name", values_to = "int")

  quant_target <- left_join(quant_target, md, by = "name")
  
  quant_target$strain_group <- factor(
    quant_target$strain_group,
    levels = c("wildtype_veg", "disfp_veg", "wildtype_stv", "disfp_stv",
               "wildtype_fbs", "disfp_fbs", "pblank", "blank", "KA", "SM5", "HL5")
  )

  plot <- ggplot(quant_target, aes(x = strain_group, y = int, group = strain_group, fill = strain_group)) +
    geom_boxplot(outlier.shape = NA, show.legend = FALSE, colour = "black", width = 0.2) +
    geom_quasirandom(show.legend = FALSE, colour = "black", width = 0.3) +
    scale_fill_manual(values = group_colors) +
    labs(x = NULL, y = "peak area") +
    guides(color = "none") +
    scale_y_log10(
      breaks = trans_breaks("log10", function(x) 10^x),
      labels = trans_format("log10", math_format(10^.x))
    ) +
    scale_x_discrete(labels = function(x) sapply(x, function(y) rename_dict[[y]])) +
    theme_minimal(base_size = 12) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = 1, vjust = 1, face = "bold", color = "grey40"),
      axis.text.x = element_blank()
    )

  return(plot)
}


In [55]:
make_structure_plot <- function(meta, wd) {
  if (is.na(meta$smiles)) {
    ggplot() +
      ggtitle("Predicted chemical structure") +
      theme_minimal() +
      scale_x_continuous(limits = c(0, 3), name = "") +
      scale_y_continuous(limits = c(0, 2), name = "") +
      theme(
        axis.line        = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.text        = element_blank(),
        axis.ticks       = element_blank()
      )
  } else {
    svg_path <- file.path(wd, "03_analysis", "formulas",
                          "light_white-background",
                          paste0(meta$feature, ".svg"))
    svg <- svgparser::read_svg(svg_path)

    ggplot() +
      theme_minimal() +
      scale_x_continuous(limits = c(0, 1), expand = c(0, 0), name = "") +
      scale_y_continuous(limits = c(0, 1), expand = c(0, 0), name = "") +
      annotation_custom(svg, xmin = 0, xmax = 1, ymin = 0, ymax = 1) +
      theme_void() +
      coord_fixed()
  }
}


In [56]:
assemble_and_save_dashboard_plot <- function(
  smiles, meta, 
  eic_plot, eic_plot_pks, structure_plot, 
  wd, rename_mode_dict, rename_change_dict, rename_stage_dict, rename_dict_pks_intersection
) {
  width_plot <- 16

  # Compose plot layout
  if (!(meta$metaChange_pks %in% c("na", "DiSfp"))) {
    if (!is.na(smiles)) {
      eic_box_plot <- (
        (eic_plot |
        (eic_plot_pks + plot_layout(widths = c(2))) | 
        (structure_plot + plot_layout(widths = c(2)))) + plot_layout(widths = c(1, 2, 2))
      )
    } else {
      eic_box_plot <- (
        (eic_plot | 
        eic_plot_pks + plot_layout(widths = c(2))) + plot_layout(widths = c(2, 3))
      )
    }
  } else {
    if (!is.na(smiles)) {
      eic_box_plot <- (
        (eic_plot | structure_plot) + plot_layout(widths = c(2, 2))
      )
      width_plot <- 12
    } else {
      eic_box_plot <- eic_plot
      width_plot <- 12
    }
  }

  # Format p-value for markdown
  x <- as.integer(sub(".*e([+-]?[0-9]+)$", "\\1", format(meta$pvalue, scientific = TRUE)))
  pval_str <- sprintf("%.2f × 10<sup>%d</sup>", round(meta$pvalue * 10^abs(x), 2), -abs(x))

  # Add annotation
  eic_box_plot <- eic_box_plot +
    plot_annotation(
      title = sprintf(
        "**m/z:** %.4f  ·  **Rt:** %.2f min (%.1f - %.1f min) · **Scan mode:** %s · **ID:** %s", 
        meta$mz, meta$rt, meta$rtMin, meta$rtMax, 
        rename_mode_dict[[meta$mode]],
        meta$feature
      ),
      subtitle = sprintf(
        "%s %s **Change:** %s in %s %s",
        ifelse(meta$change != "abs", paste0("**p-value:** ", pval_str, " · "), ""),
        ifelse(meta$change != "abs", paste0("**log2FC:** ", round(meta$log2FC, 2), " · "), ""),
        paste0(" ", rename_change_dict[[meta$change]]),
        rename_stage_dict[[meta$stage]],
        ifelse(meta$metaChange_pks != "na", paste0("", rename_dict_pks_intersection[[meta$metaChange_pks]]), "")
      ),
      caption = sprintf(
        "%s%s%s%s%s%s%s",
        ifelse(!is.na(meta$Sirius_Formula), paste0("**Molecular Formula:** ", meta$Sirius_Formula), ""),
        ifelse(meta$NPC_Pathway != "unknown", paste0("<br><br>**NPClassifier:** ", meta$NPC_Pathway), ""),
        ifelse(meta$NPC_Superclass != "unknown", paste0(", ", meta$NPC_Superclass), ""),
        ifelse(meta$NPC_Class != "unknown", paste0(", ", meta$NPC_Class), ""),
        ifelse(meta$CF_Superclass != "unknown", paste0("  ·  **ClassyFire:** ", meta$CF_Superclass), ""),
        ifelse(meta$CF_MostSpecific != "unknown", paste0(", ", meta$CF_MostSpecific), ""),
        ifelse(meta$Sirius_Name != "unknown", paste0(" · **Predicted Name:** ", meta$Sirius_Name), "")
      )
    ) & 
    theme(
      plot.title = element_markdown(size = 12), 
      plot.subtitle = element_markdown(size = 10), 
      plot.caption = element_markdown(size = 10)
    )

  # Filename and path
  filename <- sprintf(
    "R%s_rt-%s_mz-%s_%s%s.png", 
    pad_number_float_string(meta$rank, 4), 
    convert_to_min_sec(meta$rt), 
    pad_number_float_string(meta$mz, 4), 
    meta$feature,
    ifelse(!is.na(smiles), "_dbhit", "")
  )

  output_path <- ifelse(
    meta$metaChange_pks != "na",
    file.path(wd, "04_results", "img", "feature_dashboards", meta$stage, meta$change,
              meta$metaChange_pks,
              ifelse(meta$change == "abs", meta$NPC_Pathway, meta$CF_Superclass),
              filename),
    file.path(wd, "04_results", "img", "feature_dashboards", meta$stage, meta$change,
              ifelse(meta$change == "abs", meta$NPC_Pathway, meta$CF_Superclass),
              filename)
  )

  # Save
  ggsave(
    filename = output_path,
    plot = eic_box_plot,
    width = width_plot,
    height = 4,
    dpi = 300,
    units = "in",
    bg = "white",
    create.dir = TRUE
  )

  return(eic_box_plot)
}


In [57]:
get_dashboard_metadata <- function(feature_selected, stage_selected, ft, ft_changes) {
  # Pull values from ft and ft_changes
  mode_selected       <- ft %>% filter(feature == feature_selected) %>% pull(mode)
  msms_selected       <- ft %>% filter(feature == feature_selected) %>% pull(msms)
  rank_selected       <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(rank)
  change_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(change)
  pks_intersection    <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(metaChange_pks)
  log2FC_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(log2FC)
  pvalue_selected     <- ft_changes %>% filter(feature == feature_selected, stage == stage_selected) %>% pull(pvalue)

  rt_selected         <- ft %>% filter(feature == feature_selected) %>% pull(rt)
  mz_selected         <- ft %>% filter(feature == feature_selected) %>% pull(mz)
  rtMin               <- ft %>% filter(feature == feature_selected) %>% pull(rt_range.min)
  rtMin_plot          <- max(rtMin - 5, 0)
  rtMin_plot          <- rtMin
  rtMax               <- ft %>% filter(feature == feature_selected) %>% pull(rt_range.max)
  rtMax_plot          <- min(rtMax + 5, 12)
  rtMax_plot          <- rtMax

  CF_Superclass       <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.superclass")
  CF_Class            <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.class")
  CF_Subclass         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.subclass")
  CF_Level5           <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.level.5")
  CF_MostSpecific     <- ft %>% filter(feature == feature_selected) %>% pull("sirius_ClassyFire.most.specific.class")

  NPC_Pathway         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.pathway")
  NPC_Superclass      <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.superclass")
  NPC_Class           <- ft %>% filter(feature == feature_selected) %>% pull("sirius_NPC.class")

  Sirius_Name         <- ft %>% filter(feature == feature_selected) %>% pull("sirius_name")
  Sirius_Formula      <- ft %>% filter(feature == feature_selected) %>% pull("sirius_molecularFormula")
  Sirius_Smiles       <- ft %>% filter(feature == feature_selected) %>% pull("sirius_smiles")

  # Return as a named list
  return(list(
    feature = feature_selected,
    stage = stage_selected,
    mode = mode_selected,
    msms = msms_selected,
    rank = rank_selected,
    change = change_selected,
    metaChange_pks = pks_intersection,
    log2FC = log2FC_selected,
    pvalue = pvalue_selected,
    rt = rt_selected,
    mz = mz_selected,
    rtMin = rtMin,
    rtMin_plot = rtMin_plot,
    rtMax = rtMax,
    rtMax_plot = rtMax_plot,
    CF_Superclass = CF_Superclass,
    CF_Class = CF_Class,
    CF_Subclass = CF_Subclass,
    CF_Level5 = CF_Level5,
    CF_MostSpecific = CF_MostSpecific,
    NPC_Pathway = NPC_Pathway,
    NPC_Superclass = NPC_Superclass,
    NPC_Class = NPC_Class,
    Sirius_Name = Sirius_Name,
    Sirius_Formula = Sirius_Formula,
    smiles = Sirius_Smiles
  ))
}


make_eic_plot_xenic <- function(feature_selected, meta, md, wd) {
  eic_path <- file.path(wd, "03_analysis", "eic", paste0(feature_selected, ".csv"))
  eic_data <- read.csv(eic_path) %>%
    rename(filename = File, rtime = RT, intensity = Intensity)

  eic_data$filename <- str_replace_all(eic_data$filename, ".mzML", "")
  eic_data <- merge(eic_data, md, by.x = "filename", by.y = "name", all.x = TRUE)

  eic_filtered <- eic_data %>%
    filter(between(rtime / 60, meta$rtMin_plot, meta$rtMax_plot)) %>%
    filter(type == "sample", cultivation == "xenic") %>%
    filter(replicate %in% c(1, 2, 3, 7, 8, 9, 13, 14, 15)) %>%
    filter(stage == meta$stage)

  eic_filtered$strain_group <- factor(
    eic_filtered$strain_group,
    levels = c("wildtype_veg", "disfp_veg", "wildtype_stv", "disfp_stv", "wildtype_fbs", "disfp_fbs", "pblank", "blank", "KA", "SM5", "HL5")
  )

  dynamic_breaks <- calculate_focused_breaks(range(eic_filtered$intensity))
  max_int <- eic_filtered %>%
    filter(between(rtime / 60, meta$rtMin, meta$rtMax)) %>%
    pull(intensity) %>% max()

  plot <- eic_filtered %>% arrange(strain_group) %>%
    ggplot(aes(x = rtime / 60, y = intensity, group = filename, fill = strain_group, color = strain_group)) +
    geom_segment(aes(x = meta$rt, xend = meta$rt, y = 0, yend = Inf), color = "darkgrey", linetype = "solid") +
    geom_segment(aes(x = meta$rtMin, xend = meta$rtMin, y = 0, yend = Inf), color = "grey80", linetype = "dashed") +
    geom_segment(aes(x = meta$rtMax, xend = meta$rtMax, y = 0, yend = Inf), color = "grey80", linetype = "dashed") +
    geom_ribbon(aes(ymin = 0, ymax = intensity), alpha = 0.2) +
    geom_line() +
    labs(x = "retention time in min", y = "intensity", color = "genotype") +
    scale_fill_manual(values = group_colors_light) +
    scale_color_manual(values = group_colors) +
    guides(color = FALSE, fill = FALSE) +
    scale_y_continuous(label = scientific_10, breaks = dynamic_breaks) +
    coord_cartesian(xlim = c(meta$rtMin_plot, meta$rtMax_plot), ylim = c(NA, max_int)) +
    theme_minimal(base_size = 12) +
    facet_wrap(~strain_group, nrow = 1, labeller = labeller) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = .99, vjust = -20, face = "bold", color = "grey40")
    )

  return(plot)
}


make_eic_plot_axenic <- function(feature_selected, meta, md, wd) {
  eic_path <- file.path(wd, "03_analysis", "eic", paste0(feature_selected, ".csv"))
  eic_data <- read.csv(eic_path) %>%
    rename(filename = File, rtime = RT, intensity = Intensity)

  eic_data$filename <- str_replace_all(eic_data$filename, ".mzML", "")
  eic_data <- merge(eic_data, md, by.x = "filename", by.y = "name", all.x = TRUE)

  eic_filtered <- eic_data %>%
    filter(between(rtime / 60, meta$rtMin_plot, meta$rtMax_plot)) %>%
    filter(type == "sample", cultivation == "axenic", strain_group != "wildtype1_axenic_fbs") %>%
    filter(replicate %in% c(1, 2, 3)) %>%
    filter(stage == "fbs")

  eic_filtered$strain_group <- factor(
    eic_filtered$strain_group,
    levels = c("wildtype2_axenic_fbs", "pks5_fbs", "pks7_fbs", "pks24_fbs", "pks25_fbs", "pks24pks25_fbs")
  )

  dynamic_breaks <- calculate_focused_breaks(range(eic_filtered$intensity))
  max_int <- eic_filtered %>%
    filter(between(rtime / 60, meta$rtMin, meta$rtMax)) %>%
    pull(intensity) %>% max()

  plot <- eic_filtered %>% arrange(strain_group) %>%
    ggplot(aes(x = rtime / 60, y = intensity, group = filename, fill = strain_group, color = strain_group)) +
    geom_vline(xintercept = meta$rt, linetype = "solid", color = "darkgrey") +
    geom_ribbon(aes(ymin = 0, ymax = intensity), alpha = 0.2) +
    geom_line() +
    labs(x = "retention time in min", y = "intensity", color = "genotype") +
    scale_fill_manual(values = group_colors_light) +
    scale_color_manual(values = group_colors) +
    guides(color = FALSE, fill = FALSE) +
    scale_y_continuous(label = scientific_10, breaks = dynamic_breaks) +
    coord_cartesian(xlim = c(meta$rtMin_plot, meta$rtMax_plot), ylim = c(NA, max_int)) +
    theme_minimal(base_size = 12) +
    facet_wrap(~strain_group, nrow = 1, labeller = labeller_pks) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = .99, vjust = -20, face = "bold", color = "grey40")
    )

  return(plot)
}


make_boxplot <- function(feature_selected, stage_selected, ft, md) {
  md.select <- md %>%
    filter(type == "sample", cultivation == "xenic", stage == stage_selected)
  
  name <- md.select %>% pull(name)

  quant_target <- ft %>%
    filter(feature == feature_selected) %>%
    .[, name] %>%
    pivot_longer(cols = name, names_to = "name", values_to = "int")

  quant_target <- left_join(quant_target, md, by = "name")
  
  quant_target$strain_group <- factor(
    quant_target$strain_group,
    levels = c("wildtype_veg", "disfp_veg", "wildtype_stv", "disfp_stv",
               "wildtype_fbs", "disfp_fbs", "pblank", "blank", "KA", "SM5", "HL5")
  )

  plot <- ggplot(quant_target, aes(x = strain_group, y = int, group = strain_group, fill = strain_group)) +
    geom_boxplot(outlier.shape = NA, show.legend = FALSE, colour = "black", width = 0.2) +
    geom_quasirandom(show.legend = FALSE, colour = "black", width = 0.3) +
    scale_fill_manual(values = group_colors) +
    labs(x = NULL, y = "peak area") +
    guides(color = "none") +
    scale_y_log10(
      breaks = trans_breaks("log10", function(x) 10^x),
      labels = trans_format("log10", math_format(10^.x))
    ) +
    scale_x_discrete(labels = function(x) sapply(x, function(y) rename_dict[[y]])) +
    theme_minimal(base_size = 12) +
    theme(
      panel.grid.major.x = element_blank(),
      panel.grid.minor.x = element_blank(),
      panel.grid.minor.y = element_blank(),
      plot.title = element_text(hjust = 1, vjust = 1, face = "bold", color = "grey40"),
      axis.text.x = element_blank()
    )

  return(plot)
}


make_structure_plot <- function(meta, wd) {
  if (is.na(meta$smiles)) {
    ggplot() +
      ggtitle("Predicted chemical structure") +
      theme_minimal() +
      scale_x_continuous(limits = c(0, 3), name = "") +
      scale_y_continuous(limits = c(0, 2), name = "") +
      theme(
        axis.line        = element_blank(),
        panel.grid.major = element_blank(),
        panel.grid.minor = element_blank(),
        axis.text        = element_blank(),
        axis.ticks       = element_blank()
      )
  } else {
    svg_path <- file.path(wd, "03_analysis", "formulas",
                          "light_white-background",
                          paste0(meta$feature, ".svg"))
    svg <- svgparser::read_svg(svg_path)

    ggplot() +
      theme_minimal() +
      scale_x_continuous(limits = c(0, 1), expand = c(0, 0), name = "") +
      scale_y_continuous(limits = c(0, 1), expand = c(0, 0), name = "") +
      annotation_custom(svg, xmin = 0, xmax = 1, ymin = 0, ymax = 1) +
      theme_void() +
      coord_fixed()
  }
}


assemble_and_save_dashboard_plot <- function(
  smiles, meta, 
  eic_plot, eic_plot_pks, structure_plot, 
  wd, rename_mode_dict, rename_change_dict, rename_stage_dict, rename_dict_pks_intersection
) {
  width_plot <- 16

  # Compose plot layout
  if (!(meta$metaChange_pks %in% c("na", "disfp"))) {
    if (!is.na(smiles)) {
      eic_box_plot <- (
        (eic_plot |
        (eic_plot_pks + plot_layout(widths = c(2))) | 
        (structure_plot + plot_layout(widths = c(2)))) + plot_layout(widths = c(1, 2, 2))
      )
    } else {
      eic_box_plot <- (
        (eic_plot | 
        eic_plot_pks + plot_layout(widths = c(2))) + plot_layout(widths = c(2, 3))
      )
    }
  } else {
    if (!is.na(smiles)) {
      eic_box_plot <- (
        (eic_plot | structure_plot) + plot_layout(widths = c(2, 2))
      )
      width_plot <- 12
    } else {
      eic_box_plot <- eic_plot
      width_plot <- 12
    }
  }

  # Format p-value for markdown
  x <- as.integer(sub(".*e([+-]?[0-9]+)$", "\\1", format(meta$pvalue, scientific = TRUE)))
  pval_str <- sprintf("%.2f × 10<sup>%d</sup>", round(meta$pvalue * 10^abs(x), 2), -abs(x))

  # Add annotation
  eic_box_plot <- eic_box_plot +
    plot_annotation(
      title = sprintf(
        "**m/z:** %.4f  ·  **Rt:** %.2f min (%.1f - %.1f min) · **Scan mode:** %s · **ID:** %s", 
        meta$mz, meta$rt, meta$rtMin, meta$rtMax, 
        rename_mode_dict[[meta$mode]],
        meta$feature
      ),
      subtitle = sprintf(
        "%s %s **Change:** %s in %s %s",
        ifelse(meta$change != "abs", paste0("**p-value:** ", pval_str, " · "), ""),
        ifelse(meta$change != "abs", paste0("**log2FC:** ", round(meta$log2FC, 2), " · "), ""),
        paste0(" ", rename_change_dict[[meta$change]]),
        rename_stage_dict[[meta$stage]],
        ifelse(meta$metaChange_pks != "na", paste0("", rename_dict_pks_intersection[[meta$metaChange_pks]]), "")
      ),
      caption = sprintf(
        "%s%s%s%s%s%s%s",
        ifelse(!is.na(meta$Sirius_Formula), paste0("**Molecular Formula:** ", meta$Sirius_Formula), ""),
        ifelse(meta$NPC_Pathway != "unknown", paste0("<br><br>**NPClassifier:** ", meta$NPC_Pathway), ""),
        ifelse(meta$NPC_Superclass != "unknown", paste0(", ", meta$NPC_Superclass), ""),
        ifelse(meta$NPC_Class != "unknown", paste0(", ", meta$NPC_Class), ""),
        ifelse(meta$CF_Superclass != "unknown", paste0("  ·  **ClassyFire:** ", meta$CF_Superclass), ""),
        ifelse(meta$CF_MostSpecific != "unknown", paste0(", ", meta$CF_MostSpecific), ""),
        ifelse(meta$Sirius_Name != "unknown", paste0(" · **Predicted Name:** ", meta$Sirius_Name), "")
      )
    ) & 
    theme(
      plot.title = element_markdown(size = 12), 
      plot.subtitle = element_markdown(size = 10), 
      plot.caption = element_markdown(size = 10)
    )

  # Filename and path
  filename <- sprintf(
    "R%s_rt-%s_mz-%s_%s%s.png", 
    pad_number_float_string(meta$rank, 4), 
    convert_to_min_sec(meta$rt), 
    pad_number_float_string(meta$mz, 4), 
    meta$feature,
    ifelse(!is.na(smiles), "_dbhit", "")
  )

  output_path <- ifelse(
    meta$metaChange_pks != "na",
    file.path(wd, "05_results", "img", "feature_dashboards", meta$stage, meta$change,
              meta$metaChange_pks,
              ifelse(meta$change == "abs", meta$NPC_Pathway, meta$CF_Superclass),
              filename),
    file.path(wd, "05_results", "img", "feature_dashboards", meta$stage, meta$change,
              ifelse(meta$change == "abs", meta$NPC_Pathway, meta$CF_Superclass),
              filename)
  )

  # Save
  ggsave(
    filename = output_path,
    plot = eic_box_plot,
    width = width_plot,
    height = 4,
    dpi = 300,
    units = "in",
    bg = "white",
    create.dir = TRUE
  )

  return(eic_box_plot)
}


### Generate the dashboards
#### For all relevant features
The plots are automatically saved to the results directory and organized by stage, change type, and compound class.


In [36]:
for (i in 1:nrow(ft_selected)) {
  feature_selected <- ft_selected[i, ] %>% pull(feature)
  stage_selected <- ft_selected[i, ] %>% pull(stage)

  meta <- get_dashboard_metadata(feature_selected, stage_selected, ft, ft_changes)
  eic_plot <- make_eic_plot_xenic(feature_selected, meta, md, wd)
  eic_plot_pks <- make_eic_plot_axenic(feature_selected, meta, md, wd)
  boxplot <- make_boxplot(feature_selected, stage_selected, ft, md)
  structure_plot <- make_structure_plot(meta, wd)
  plot <- assemble_and_save_dashboard_plot(
    meta$smiles, meta, 
    eic_plot, eic_plot_pks, structure_plot, 
    wd, rename_mode_dict, rename_change_dict, rename_stage_dict, rename_dict_pks_intersection
  )
}


#### For specified groups of features as presented in the SI

In [204]:
#Define groups of features
custom_export_dict <- list(
  "01_veg"        = c("pos_189242", "pos_172301", "pos_171600", "neg_090313", "neg_099335", "pos_145005", "pos_158988", "neg_086194", "neg_049822", "neg_026464", "neg_030700"),
  "02_stv"        = c("neg_115043", "neg_113918", "neg_110478", "neg_078140", "neg_101855", "neg_116340", "neg_110750"),
  "03_lipdis"     = c("neg_091054", "neg_095250", "neg_095782", "neg_086194", "neg_026464", "neg_116252", "neg_100813"),
  "04_pks5"       = c("pos_168395", "pos_156645", "neg_087773", "neg_090312", "neg_090313"),
  "05_pks7"       = c("neg_065634", "neg_066974", "neg_067194"),
  "06_pks24pks25" = c("pos_210911", "pos_229743", "neg_132513", "neg_150037", "neg_087049")
)


In [ ]:
# Assemble figures
assemble_and_return_dashboard_plot_extended <- function(
  smiles, meta, 
  eic_plot, eic_plot_pks, structure_plot, 
  wd, rename_mode_dict, rename_change_dict, rename_stage_dict, rename_dict_pks_intersection
) {
  width_plot <- 16

  # Format p-value
  x <- as.integer(sub(".*e([+-]?[0-9]+)$", "\\1", format(meta$pvalue, scientific = TRUE)))
  pval_str <- sprintf("%.2f × 10<sup>%d</sup>", round(meta$pvalue * 10^abs(x), 2), -abs(x))

  # Compose dashboard plot
  if (!(meta$metaChange_pks %in% c("na", "disfp"))) {
    if (!is.na(smiles)) {
      dashboard <- (
        eic_plot |
        (eic_plot_pks + plot_layout(widths = c(2))) |
        (structure_plot + plot_layout(widths = c(2)))
      ) + plot_layout(widths = c(1, 3, 1))
    } else {
      dashboard <- (
        eic_plot |
        (eic_plot_pks + plot_layout(widths = c(2)))
      ) + plot_layout(widths = c(2, 3))
    }
  } else {
    if (!is.na(smiles)) {
      dashboard <- (
        eic_plot | structure_plot
      ) + plot_layout(widths = c(2, 2))
      width_plot <- 12
    } else {
      dashboard <- eic_plot
      width_plot <- 12
    }
  }



  # Add annotation
  dashboard <- dashboard +
    plot_annotation(
      title = sprintf(
        "**m/z:** %.4f  ·  **Rt:** %.2f min (%.1f - %.1f min) · **Scan mode:** %s · **ID:** %s", 
        meta$mz, meta$rt, meta$rtMin, meta$rtMax, 
        rename_mode_dict[[meta$mode]], meta$feature
      ),
      subtitle = sprintf(
        "%s %s **Change:** %s in %s %s",
        ifelse(meta$change != "abs", paste0("**p-value:** ", pval_str, " · "), ""),
        ifelse(meta$change != "abs", paste0("**log2FC:** ", round(meta$log2FC, 2), " · "), ""),
        paste0(" ", rename_change_dict[[meta$change]]),
        rename_stage_dict[[meta$stage]],
        ifelse(meta$metaChange_pks != "na", paste0("", rename_dict_pks_intersection[[meta$metaChange_pks]]), "")
      ),
      caption = sprintf(
        "%s%s%s%s%s%s%s%s",
        ifelse(!is.na(meta$Sirius_Formula), paste0("**Molecular Formula:** ", meta$Sirius_Formula), ""),        
        ifelse(meta$Sirius_Name != "unknown", paste0(" · **Predicted Name:** ", meta$Sirius_Name), ""),
        ifelse(meta$NPC_Pathway != "unknown", paste0("<br><br>**NPClassifier:** ", meta$NPC_Pathway), ""),
        ifelse(meta$NPC_Superclass != "unknown", paste0(", ", meta$NPC_Superclass), ""),
        ifelse(meta$NPC_Class != "unknown", paste0(", ", meta$NPC_Class), ""),
        ifelse(meta$CF_Superclass != "unknown", paste0("  ·  **ClassyFire:** ", meta$CF_Superclass), ""),
        ifelse(meta$CF_MostSpecific != "unknown", paste0(", ", meta$CF_MostSpecific), ""),
        "<br><br>"
      )
    ) &
    theme(
      plot.title = element_markdown(size = 12, margin = margin(l = 40, t = 10)),
      plot.subtitle = element_markdown(size = 10, margin = margin(l = 40, t = 10)),
      plot.caption = element_markdown(size = 10)
    )


  return(dashboard)
}


Export the figues with tagged subplots

In [ ]:
tag_letters <- LETTERS  # A, B, C, ...

for (export_label in names(custom_export_dict)) {
  features <- custom_export_dict[[export_label]]
  dashboards <- list()

  for (i in seq_along(features)) {
    feature <- features[i]

    stage <- ft_selected %>%
      filter(feature == !!feature) %>%
      pull(stage) %>%
      first()

    if (is.na(stage)) next

    meta <- get_dashboard_metadata(feature, stage, ft, ft_changes)

    dashboard_plot <- assemble_and_return_dashboard_plot_extended(
      meta$smiles, meta,
      make_eic_plot_xenic(feature, meta, md, wd),
      make_eic_plot_axenic(feature, meta, md, wd),
      make_structure_plot(meta, wd),
      wd, rename_mode_dict, rename_change_dict, rename_stage_dict, rename_dict_pks_intersection
    )

    dashboards[[i]] <- wrap_elements(full = dashboard_plot)
  }

  # Chunk into groups of 4
  chunk_size <- 4
  n_chunks <- ceiling(length(dashboards) / chunk_size)

  for (chunk_index in seq_len(n_chunks)) {
    start <- (chunk_index - 1) * chunk_size + 1
    end <- min(chunk_index * chunk_size, length(dashboards))
    chunk_dashboards <- dashboards[start:end]

    # Select the right tag letters
    tag_subset <- tag_letters[start:end]

    # Combine chunk into taggable patchwork
    patch <- wrap_plots(chunk_dashboards, ncol = 1) +
      plot_annotation(tag_levels = list(tag_subset)) &  # manual tag letters
      theme(plot.tag = element_text(size = 18, face = "bold"))

    # Save chunk
    ggsave(
      filename = file.path(
        wd, "04_results", "img", "feature_dashboards",
        "export_examples",
        sprintf("Dashboards_%s_part%02d.png", export_label, chunk_index)
      ),
      plot = patch,
      create.dir = TRUE,
      width = 14,
      height = 4 * length(chunk_dashboards),
      dpi = 300,
      units = "in",
      bg = "white"
    )
  }
}
